# Khai bao

In [33]:
import sys
sys.path.append("../Common/")

import TradeSSI

# Get Data

In [34]:
symbol = 'STB'
from_date = '01/09/2025'
to_date = '01/10/2025'
data = TradeSSI.TradeSSI.get_daily_OHLC_Data(symbol, from_date, to_date)
data

,Symbol,Market,TradingDate,Time,Open,High,Low,Close,Volume,Value
0,STB,HOSE,03/09/2025,None,56000,56300,55200,56300,7512300,418357710000
1,STB,HOSE,04/09/2025,None,56700,56800,55700,56400,7720900,433564380000
2,STB,HOSE,05/09/2025,None,56900,57100,54800,56000,11299600,634152140000
3,STB,HOSE,08/09/2025,None,55700,56200,53900,54000,11396400,625589500000
4,STB,HOSE,09/09/2025,None,53900,54300,53100,54200,6237400,335474740000
5,STB,HOSE,10/09/2025,None,54500,54900,53400,53900,7469300,403811520000
6,STB,HOSE,11/09/2025,None,54200,55700,52600,55400,12854900,697499140000
7,STB,HOSE,12/09/2025,None,55900,55900,55100,55400,6397200,354938830000
8,STB,HOSE,15/09/2025,None,55200,56000,54500,56000,8931100,493185510000
9,STB,HOSE,16/09/2025,None,56400,56900,55900,56100,6628000,372779440000


# Process + Chien luoc cua chung khoan: Buy/ Sell
### MA10 > MA20 thi Buy, MA10 < MA20 thi Sell

In [35]:
# Chien luoc
import pandas as pd
import talib

# Chuyen cot Close thanh float
# Sửa lỗi parse ngày tháng - sử dụng format đúng cho DD/MM/YYYY
data['TradingDate'] = pd.to_datetime(data['TradingDate'], format='%d/%m/%Y')
data['Open'] = data['Open'].astype(float)
data['High'] = data['High'].astype(float)
data['Low'] = data['Low'].astype(float)
data['Close'] = data['Close'].astype(float)
data['Volume'] = data['Volume'].astype(float)

# Tinh MA10 va MA20
data['MA10'] = talib.SMA(data['Close'], timeperiod=10)
data['MA20'] = talib.SMA(data['Close'], timeperiod=20)

# Chi lay cac cot Open, High, Low, Close, Volume, MA10, MA20
data = data[['Open', 'High', 'Low', 'Close', 'Volume', 'MA10', 'MA20']]

# Tao cot Buy/ Sell
data['Buy_Signal'] = (data['MA10'] > data['MA20'])
data['Sell_Signal'] = (data['MA10'] < data['MA20'])

# In ra cot Buy/ Sell
data

C:\Users\TGC\AppData\Local\Temp\ipykernel_2668\3759342669.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Buy_Signal'] = (data['MA10'] > data['MA20'])
C:\Users\TGC\AppData\Local\Temp\ipykernel_2668\3759342669.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Sell_Signal'] = (data['MA10'] < data['MA20'])


,Open,High,Low,Close,Volume,MA10,MA20,Buy_Signal,Sell_Signal
0,56000.0,56300.0,55200.0,56300.0,7512300.0,NaN,NaN,False,False
1,56700.0,56800.0,55700.0,56400.0,7720900.0,NaN,NaN,False,False
2,56900.0,57100.0,54800.0,56000.0,11299600.0,NaN,NaN,False,False
3,55700.0,56200.0,53900.0,54000.0,11396400.0,NaN,NaN,False,False
4,53900.0,54300.0,53100.0,54200.0,6237400.0,NaN,NaN,False,False
5,54500.0,54900.0,53400.0,53900.0,7469300.0,NaN,NaN,False,False
6,54200.0,55700.0,52600.0,55400.0,12854900.0,NaN,NaN,False,False
7,55900.0,55900.0,55100.0,55400.0,6397200.0,NaN,NaN,False,False
8,55200.0,56000.0,54500.0,56000.0,8931100.0,NaN,NaN,False,False
9,56400.0,56900.0,55900.0,56100.0,6628000.0,55370.0,NaN,False,False


In [36]:
import redis
from datetime import datetime
import numpy as np

r = redis.Redis(host='localhost', port=6379, db=0) # 0-5 CK, 6-10 Crypto, 11-15 Forex
# Tạo hash key
hash_key = 'Buoi 7_OG'
last_record = data.iloc[-1].copy()  # Sử dụng .copy() để tránh warning

# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
    # Thêm Insertdate vào record trước khi lưu
    last_record['Insertdate'] = datetime.now().isoformat()
    
    for field, value in last_record.to_dict().items():
        # Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
        if isinstance(value, pd.Timestamp):
            value = value.isoformat()
        elif isinstance(value, (int, np.uint64)):
            value = str(value)
        r.hset(hash_key, field, value)  
        r.hset(hash_key, 'Symbol', symbol)
    
    print("Tín hiệu giao dịch:")
    print(last_record)   
else:
    print("Không có tín hiệu giao dịch:")
    print(last_record)   
    print('Không có tín hiệu!')

Tín hiệu giao dịch:
Open                              57100.0
High                              59800.0
Low                               56700.0
Close                             59800.0
Volume                         18440100.0
MA10                              56610.0
MA20                              55985.0
Buy_Signal                           True
Sell_Signal                         False
Insertdate     2025-10-01T22:04:59.864185
Name: 20, dtype: object
